## Chirps Data

In [1]:
# conda install geopandas rasterio rasterstats -c conda-forge

import os
import re
import glob
import pandas as pd
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from rasterstats import zonal_stats
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns
from scipy.stats import zscore


In [2]:

def extract_monthly_precipitation(data_folder, shapefile_path, var_name='precip'):
    adm2_gdf = gpd.read_file(shapefile_path).to_crs("EPSG:4326")
    
    results = {name: [] for name in adm2_gdf['GID_2']}
    months = []
    all_data = []  # To store all raster data for anomaly mapping

    for filename in sorted(os.listdir(data_folder)):
        filepath = os.path.join(data_folder, filename)
        month_data = None

        if filename.endswith(('.tif', '.tiff')):
            month_match = re.search(r'(\d{2})(?=\.\w+$)', filename)
            if not month_match:
                print(f"Skipping {filename}: no month found.")
                continue
            month = month_match.group(1)
            months.append(month)

            # Read raster data
            da = rxr.open_rasterio(filepath).squeeze().drop_vars('band')
            month_data = da
            
            stats = zonal_stats(adm2_gdf, filepath, stats="mean", nodata=-9999)
            
            for i, name in enumerate(adm2_gdf['GID_2']):
                mean_val = stats[i]['mean'] if stats[i]['mean'] is not None else 0
                results[name].append(mean_val)

        elif filename.endswith('.nc'):
            match = re.search(r'(\d{6})', filename)
            if not match:
                print(f"Skipping {filename}: cannot extract date.")
                continue
            month = match.group(1)[-2:]
            months.append(month)

            ds = xr.open_dataset(filepath)
            if var_name not in ds:
                print(f"{var_name} not found in {filename}")
                continue

            var = ds[var_name]
            if "time" in var.dims:
                var = var.isel(time=0)  # if time dimension exists, take first time step

            var.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
            var.rio.write_crs("EPSG:4326", inplace=True)
            month_data = var

            temp_tif = "temp.nc2tif.tif"
            var.rio.to_raster(temp_tif)

            stats = zonal_stats(adm2_gdf, temp_tif, stats="mean", nodata=-9999)
            
            for i, name in enumerate(adm2_gdf['GID_2']):
                mean_val = stats[i]['mean'] if stats[i]['mean'] is not None else 0
                results[name].append(mean_val)

            os.remove(temp_tif)

        else:
            print(f"Skipping {filename}: unsupported format.")
            continue
        
        if month_data is not None:
            all_data.append(month_data)

    # Create DataFrame with months as index and admin2 names as columns
    result_df = pd.DataFrame.from_dict(results, orient='index').T
    result_df.index = [f"precip_{month}" for month in months]
    
    # Add year if available from filename pattern
    year_match = re.search(r'(\d{4})', os.path.basename(data_folder))
    if year_match:
        year = year_match.group(1)
        result_df.index = [f"{month}_{year}" for month in result_df.index]

    csv_path = os.path.join(data_folder, f"monthly_precipitation_{year}.csv")
    result_df.to_csv(csv_path)

    print(f"Saved monthly precipitation data to {csv_path}")
    
    # Create anomaly maps if we have multiple time periods
    if len(all_data) > 1:
        create_anomaly_maps(data_folder, all_data, months, adm2_gdf, year if 'year' in locals() else None)
    
    return result_df

def create_anomaly_maps(data_folder, all_data, months, adm_gdf, year=None):
    """Create spatial anomaly maps clipped to admin area extent"""
    # Stack all data along a new dimension and calculate mean
    stacked = xr.concat(all_data, dim='time')
    mean_data = stacked.mean(dim='time')
    
    # Get the bounding box of admin areas
    minx, miny, maxx, maxy = adm_gdf.total_bounds
    
    anomaly_folder = os.path.join(data_folder, 'anomaly_maps')
    os.makedirs(anomaly_folder, exist_ok=True)
    
    # Create anomaly map for each month
    for i, (month, data) in enumerate(zip(months, all_data)):
        anomaly = data - mean_data
        print(anomaly.coords)
        print(anomaly.dims)
        
        # Determine coordinate names (handle different dimension names)
        x_dim = 'lon' if 'lon' in anomaly.dims else 'x'
        y_dim = 'lat' if 'lat' in anomaly.dims else 'y'
        
        try:
            # Clip to admin area extent
            anomaly = anomaly.sel(
                **{x_dim: slice(minx, maxx),
                y_dim: slice(maxy, miny)  # Reversed for correct orientation
            })
        except KeyError as e:
            print(f"Could not clip data: {e}")
            print(f"Available dimensions: {anomaly.dims}")
            continue
        
        # Calculate symmetric color limits
        max_abs = float(np.nanmax(np.abs(anomaly.values)))
        
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # Plot the anomaly data
        im = anomaly.plot(ax=ax, cmap='RdBu', 
                         vmin=-max_abs, 
                         vmax=max_abs,
                         add_colorbar=False)
        
        # Add colorbar
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        plt.colorbar(im, cax=cax, label='Precipitation Anomaly (mm)')
        
        title = f"Precipitation Anomaly - Month {month}"
        if year:
            title += f" ({year})"
        ax.set_title(title)
        
        # Remove axis frame
        ax.set_frame_on(False)
        ax.set_xticks([])
        ax.set_yticks([])
        
        # Save figure
        filename = f"anomaly_map_{month}"
        if year:
            filename += f"_{year}"
        filename = os.path.join(anomaly_folder, f"{filename}.png")
        plt.savefig(filename, bbox_inches='tight', dpi=300)
        plt.close()
    
    print(f"Saved anomaly maps to {anomaly_folder}/")


def plot_monthly_precip_per_adm(csv_path, output_dir="adm2_plots"):
    df = pd.read_csv(csv_path, index_col=0)
    
    # Extract year from index if available
    year = df.index[0].split('_')[-1] if '_' in df.index[0] else "unknown_year"
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Create individual plots for each admin area
    for admin in df.columns:
        plt.figure(figsize=(10, 5))
        plt.plot(df.index, df[admin], marker='o', label=admin)
        plt.title(f"Monthly Precipitation - {admin} ({year})")
        plt.xlabel("Month")
        plt.ylabel("Precipitation (mm)")
        plt.xticks(rotation=45)
        plt.grid(True)
        plt.tight_layout()
        
        filename = os.path.join(output_dir, f"{admin.replace(' ', '_')}_{year}_ppt.png")
        plt.savefig(filename)
        plt.close()
    
    
    print(f"Saved plots to {output_dir}/")

In [3]:
# df_shp = gpd.read_file(r"D:\PHD\PLUS-CLIMB\gadm41_SEN_shp\gadm41_SEN_2.shp")
# df_shp
# df_shp["GID_2"] = df_shp["GID_2"].apply(lambda x: f"SN{int(x.split('.')[1]):02}{int(x.split('.')[2].split('_')[0]):02}")
# df_shp
# df_shp.to_file("updated_adm_2.shp")



In [4]:
data_folder = r"D:\PHD\CLIMB\Data\dataset\chirps"

for year in [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]:
    year = str(year)
    print(f"Processing data for {year}...")
    
    # Extract monthly precipitation data
    extract_monthly_precipitation(
        data_folder=os.path.join(data_folder, year),
        shapefile_path=r"updated_adm_2.shp",
        var_name=None
    )
    plot_monthly_precip_per_adm(
        os.path.join(data_folder, year, f"monthly_precipitation_{year}.csv"),
        output_dir=os.path.join(data_folder, year, "adm2_plots")
    )

Processing data for 2010...
Skipping adm2_plots: unsupported format.
Skipping anomaly_maps: unsupported format.


c:\Users\Khizer Zakir\.conda\envs\geo_env\lib\site-packages\pyogrio\core.py:36: RuntimeWarning: Could not detect PROJ data files.  Set PROJ_LIB environment variable to the correct path.
  _init_proj_data()


Skipping monthly_precipitation_2010.csv: unsupported format.
Saved monthly precipitation data to D:\PHD\CLIMB\Data\dataset\chirps\2010\monthly_precipitation_2010.csv
Coordinates:
  * x            (x) float64 12kB -19.97 -19.92 -19.87 ... 54.88 54.93 54.98
  * y            (y) float64 13kB 39.97 39.92 39.87 ... -39.88 -39.93 -39.98
    spatial_ref  int32 4B 0
('y', 'x')
Coordinates:
  * x            (x) float64 12kB -19.97 -19.92 -19.87 ... 54.88 54.93 54.98
  * y            (y) float64 13kB 39.97 39.92 39.87 ... -39.88 -39.93 -39.98
    spatial_ref  int32 4B 0
('y', 'x')
Coordinates:
  * x            (x) float64 12kB -19.97 -19.92 -19.87 ... 54.88 54.93 54.98
  * y            (y) float64 13kB 39.97 39.92 39.87 ... -39.88 -39.93 -39.98
    spatial_ref  int32 4B 0
('y', 'x')
Coordinates:
  * x            (x) float64 12kB -19.97 -19.92 -19.87 ... 54.88 54.93 54.98
  * y            (y) float64 13kB 39.97 39.92 39.87 ... -39.88 -39.93 -39.98
    spatial_ref  int32 4B 0
('y', 'x')
Coordinate

In [5]:
csv_files = glob.glob(os.path.join(data_folder, "*", "monthly_precipitation_*.csv"))
csv_files.sort()
csv_files

df_concat = pd.concat([pd.read_csv(f) for f in csv_files ], ignore_index=True)
df_concat
df_concat.to_csv(os.path.join(data_folder, "monthly_precipitation_all_years.csv"), index=False)

In [6]:
# Load the CSV file to inspect the structure
file_path = os.path.join(data_folder, "monthly_precipitation_all_years.csv")
df = pd.read_csv(file_path)

# Show the first few rows and column info
df.head(), df.columns


(       Unnamed: 0    SN0101  SN0102    SN0103    SN0104    SN0201    SN0202   
 0  precip_01_2010  0.000000     0.0  0.080918  0.019503  0.005348  0.048328  \
 1  precip_02_2010  0.069417     0.0  0.354702  0.115024  0.129789  0.132225   
 2  precip_03_2010  0.000000     0.0  0.000000  0.000000  0.000000  0.000000   
 3  precip_04_2010  0.000000     0.0  0.048444  0.034354  0.039844  0.005045   
 4  precip_05_2010  2.063941     0.0  1.987570  1.631217  0.883808  0.991159   
 
      SN0203    SN0301    SN0302  ...     SN1201     SN1202    SN1203   
 0  0.209222  0.154185  0.481689  ...   0.009323   0.009313  0.086051  \
 1  0.091338  0.052876  0.003372  ...   0.078585   0.113080  0.127658   
 2  0.000000  0.000000  0.000000  ...   0.002367   0.113804  0.000514   
 3  0.011147  0.016059  0.129833  ...   1.689099   0.722078  0.043252   
 4  0.981607  0.692897  1.464581  ...  30.577966  15.067351  0.486512   
 
       SN1204    SN1301    SN1302    SN1303    SN1401    SN1402    SN1403  
 0